# Token merging vs early exit: exhaustive study across hardware regimes (frozen ViT-Tiny, CIFAR-10)

**Question:** when does token merging help, when does it not, and when do MACs pick the right design?

**Why this is cheap:** the backbone is frozen, so a head depends only on the tokens before it. One tree-shaped pass caches
CLS features for every merge schedule, heads train on those features, and every design is then evaluated offline.

**Methods compared:** full model (B0, with linear and MLP heads), exit only, merge only, exit + merge.
**Regimes:** GPU at batch 1 / 32 / 128 and CPU at batch 1 / 32, each with its own measured cost table.

**Stages** (each is resumable; choose with `GV_STAGES`): `extract, train, points, energy, select, real, report`.
To add a second machine, copy `results/` there and run `GV_STAGES=energy,real,report`.

**Rules:** designs are selected on validation only. Test numbers are stored for reporting, never for choosing.
Energy columns are never comparable across meters: check `meter` in `results/energy_tables.json` (`J` or wall-clock `s`).

## 0. Setup

In [ ]:
!pip install -q timm nvidia-ml-py matplotlib

In [ ]:
import os, json, time, math, copy, re, gzip, random, hashlib, zipfile, itertools, threading, platform
from contextlib import nullcontext
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.func import functional_call, stack_module_state, vmap
import timm

def _env(name, default, cast=int):
    return cast(os.environ[name]) if name in os.environ else default

SMOKE = os.environ.get("SMOKE_TEST", "0") == "1"      # synthetic data + random weights, for testing the code only
ON_KAGGLE = os.path.isdir("/kaggle/working")
STAGES = os.environ.get("GV_STAGES", "extract,train,points,energy,select,real,report").split(",")

DATA_DIR = os.environ.get("CIFAR_DIR", "/kaggle/input/datasets/ayush1220/cifar10/cifar10" if ON_KAGGLE else "data/cifar-10")
WORK = os.environ.get("GV_WORK", "/kaggle/working" if ON_KAGGLE else "outputs")
RESULTS = os.path.join(WORK, "results")
# Large and regenerable, so kept out of Kaggle's saved output. Locally it must NOT be /tmp if /tmp is RAM-backed.
SCRATCH = os.environ.get("GV_SCRATCH", "/tmp/gv_scratch" if ON_KAGGLE else os.path.join(WORK, "scratch"))
FEATS, SIGS = os.path.join(SCRATCH, "features"), os.path.join(SCRATCH, "sigs")
CACHE_DIR = os.path.join(SCRATCH, "cache")
for d in (RESULTS, FEATS, SIGS, CACHE_DIR):
    os.makedirs(d, exist_ok=True)

# Notebook output is hidden until the end under nbconvert, so progress also goes to a file: tail -f results/progress.log
_print = print
def print(*args, **kw):
    _print(*args, **kw)
    with open(os.path.join(RESULTS, "progress.log"), "a") as f:
        f.write(time.strftime("%H:%M:%S ") + " ".join(str(x) for x in args) + "\n")

BACKBONE = "vit_tiny_patch16_224"
IMG_SIZE = 64 if SMOKE else 224
NP = (IMG_SIZE // 16) ** 2            # patch tokens (196 at 224px)
NUM_CLASSES, SEED = 10, 42
CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

# ---- design space ----
EXIT_POOL = [2, 4, 6, 8, 10]
ARCHS = {"lin": {"arch": "linear"}, "mlp128": {"arch": "mlp", "hidden": 128}, "mlp192": {"arch": "mlp", "hidden": 192}}
RATIOS = [0.25, 0.5]                 # fraction of patch tokens removed by one merge
MAX_EXITS, MAX_MERGES = 2, 2
TAUS = [0.0, 0.02, 0.05, 0.08, 0.12, 0.16, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
EPS_LIST = [0.0025, 0.005, 0.01, 0.02]   # accuracy tolerances against B0
MIN_MERGE_USE = 0.05                     # exit+merge only counts if at least 5% of samples actually reach the merge
FINALIST_EPS = (0.005, 0.01)             # tolerances whose picks get real energy runs

# ---- data and training ----
TRAIN_PC = _env("GV_TRAIN_PC", 40 if SMOKE else 2000)   # per class (2000 -> 20k; up to 4500)
CAL_PC   = _env("GV_CAL_PC", 10 if SMOKE else 100)
VAL_PC   = _env("GV_VAL_PC", 20 if SMOKE else 400)
TEST_N   = _env("GV_TEST_N", 200 if SMOKE else 10000)
N_SEEDS  = _env("GV_SEEDS", 2 if SMOKE else 5)
EPOCHS   = _env("GV_EPOCHS", 2 if SMOKE else 10)
LR, BATCH = 2e-3, 128
EXTRACT_BATCH = _env("GV_EXTRACT_BATCH", 128)           # lower this if the GPU runs out of memory
B0_MIN_VAL_ACC = 0.90                                   # about 0.93 is expected; enforced at full data budget only

# ---- measurement ----
GPU_BATCHES = [1, 32, 128]
CPU_BATCHES = [1, 32]
GRID = {"cuda": [1, 8, 32, 128], "cpu": [1, 8, 32]}      # batch sizes in the cost tables (interpolated for shrinking batches)
WINDOW_S   = _env("GV_WINDOW", 0.05 if SMOKE else 5.0, float)
REPEATS    = _env("GV_REPEATS", 2 if SMOKE else 3)
IDLE_S     = _env("GV_IDLE", 0.1 if SMOKE else 20.0, float)
REAL_N     = _env("GV_REAL_N", 100 if SMOKE else 1000)
COOL_TO_C  = _env("GV_COOL_TO_C", 200 if SMOKE else 75, float)   # wait for the GPU to cool to this before measuring
COOL_TIMEOUT_S = _env("GV_COOL_TIMEOUT", 600)
HW_NAME    = os.environ.get("GV_HW", "")                 # optional label for this machine
USE_AMP    = os.environ.get("GV_AMP", "1") != "0"        # 0: run the GPU in float32. Measure both: on a GTX 1650 autocast is about 3.4x slower.
RUN_CPU_REGIME = os.environ.get("GV_CPU_REGIME", "auto") # auto: only when CPU energy is readable (RAPL); 1: force; 0: never
CARBON_KG_PER_KWH = 0.4                                  # ASSUMPTION: state your grid's figure in the paper

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, "| stages:", STAGES, "| smoke:", SMOKE)
if DEV == "cuda":
    print("gpu:", torch.cuda.get_device_name(0), "| memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

def amp_ctx(dev):
    return torch.autocast("cuda", dtype=torch.float16) if dev == "cuda" and USE_AMP else nullcontext()

def sync(dev):
    if dev == "cuda":
        torch.cuda.synchronize()

def save_json(obj, name):
    tmp = os.path.join(RESULTS, name + ".tmp")
    json.dump(obj, open(tmp, "w"), indent=1)
    os.replace(tmp, os.path.join(RESULTS, name))

def load_json(name, default=None):
    p = os.path.join(RESULTS, name)
    return json.load(open(p)) if os.path.exists(p) else default

## 1. Data

In [ ]:
CIFAR_URL = "https://www.cs.toronto.edu/~kriz/cifar-10-binary.tar.gz"
CIFAR_MD5 = "c32a1d4ab5d03f1284b67883e8d87530"

def read_cifar_tar(path):
    # Official binary format: 1 label byte + 3072 image bytes per record. Raw bytes, so no pickle is involved.
    import tarfile
    def part(tf, names):
        raw = b"".join(tf.extractfile(m).read() for m in sorted(names))
        a = np.frombuffer(raw, dtype=np.uint8).reshape(-1, 3073)
        return torch.from_numpy(a[:, 1:].reshape(-1, 3, 32, 32).copy()), torch.from_numpy(a[:, 0].astype(np.int64))
    with tarfile.open(path) as tf:
        names = tf.getnames()
        Xtr, Ytr = part(tf, [n for n in names if "data_batch_" in n])
        Xte, Yte = part(tf, [n for n in names if "test_batch" in n])
    assert len(Ytr) == 50000 and len(Yte) == 10000, "unexpected CIFAR-10 archive contents"
    return Xtr, Ytr, Xte, Yte

def read_split_folder(split_dir):
    from PIL import Image
    from torchvision import datasets
    ds = datasets.ImageFolder(split_dir)
    assert list(ds.classes) == CLASS_NAMES, f"class order differs: {ds.classes}"
    X = np.empty((len(ds), 32, 32, 3), dtype=np.uint8); Y = np.empty(len(ds), dtype=np.int64)
    for i, (p, y) in enumerate(ds.samples):
        X[i] = np.asarray(Image.open(p).convert("RGB")); Y[i] = y
    return torch.from_numpy(X).permute(0, 3, 1, 2).contiguous(), torch.from_numpy(Y)

def find_split_dir(root, name):
    for dirpath, _, _ in os.walk(root):
        if os.path.basename(dirpath) == name:
            return dirpath
    raise FileNotFoundError(f"no '{name}' directory under {root}")

def load_cifar():
    if SMOKE:
        g = torch.Generator().manual_seed(0)
        def synth(n):
            Y = torch.arange(n) % NUM_CLASSES
            X = (torch.rand(n, 3, 32, 32, generator=g) * 80 + Y.view(-1, 1, 1, 1) * 15).clamp(0, 255).to(torch.uint8)
            return X, Y.long()
        return (*synth(2000), *synth(max(TEST_N, 200)))
    cache = os.path.join(CACHE_DIR, "cifar10_uint8.pt")
    if os.path.exists(cache):
        d = torch.load(cache, weights_only=True)
        return d["Xtr"], d["Ytr"], d["Xte"], d["Yte"]
    tar = os.path.join(DATA_DIR, "cifar-10-binary.tar.gz")
    if os.path.isfile(tar):
        Xtr, Ytr, Xte, Yte = read_cifar_tar(tar)
    elif os.path.isdir(DATA_DIR):
        Xtr, Ytr = read_split_folder(find_split_dir(DATA_DIR, "train"))
        Xte, Yte = read_split_folder(find_split_dir(DATA_DIR, "test"))
    else:
        import urllib.request
        tar = os.path.join(CACHE_DIR, "cifar-10-binary.tar.gz")
        print("downloading CIFAR-10 (162 MB)")
        urllib.request.urlretrieve(CIFAR_URL, tar)
        assert hashlib.md5(open(tar, "rb").read()).hexdigest() == CIFAR_MD5, "CIFAR-10 checksum mismatch"
        Xtr, Ytr, Xte, Yte = read_cifar_tar(tar)
    torch.save({"Xtr": Xtr, "Ytr": Ytr, "Xte": Xte, "Yte": Yte}, cache)
    return Xtr, Ytr, Xte, Yte

def stratified(labels, per_class, seed, exclude=()):
    rng, out, ex = np.random.RandomState(seed), [], set(exclude)
    for c in range(NUM_CLASSES):
        ids = np.array([i for i in np.where(labels == c)[0] if i not in ex])
        rng.shuffle(ids)
        out.extend(ids[:per_class].tolist())
    return out

# Weights expect 0.5/0.5; asserted against timm's own config in make_backbone.
MODEL_MEAN, MODEL_STD = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)

def prepare(x_uint8, dev):
    x = x_uint8.to(dev).float().div_(255.0)
    x = F.interpolate(x, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
    m = torch.tensor(MODEL_MEAN, device=dev).view(1, 3, 1, 1); s = torch.tensor(MODEL_STD, device=dev).view(1, 3, 1, 1)
    return (x - m) / s

## 2. Model pieces and the design space

In [ ]:
def make_backbone(dev):
    kw = {"img_size": IMG_SIZE} if SMOKE else {}
    if SMOKE:
        torch.manual_seed(SEED)     # random weights must be identical in every stage, or features and real runs disagree
    bb = timm.create_model(BACKBONE, pretrained=not SMOKE, num_classes=0, **kw).to(dev).eval()
    assert np.allclose(bb.pretrained_cfg["mean"], MODEL_MEAN) and np.allclose(bb.pretrained_cfg["std"], MODEL_STD), \
        f"normalisation mismatch: weights expect {bb.pretrained_cfg['mean']} / {bb.pretrained_cfg['std']}"
    for p in bb.parameters():
        p.requires_grad = False
    return bb

def embed(bb, x):
    t = bb._pos_embed(bb.patch_embed(x))
    return bb.norm_pre(t) if hasattr(bb, "norm_pre") else t

def merge_pairs(Np, ratio):
    if ratio <= 0 or Np < 4:
        return 0
    return int(min(max(1, int(Np * ratio)), (Np + 1) // 2 - 1))

def merged_count(Np, ratio):
    return Np - merge_pairs(Np, ratio)

def token_merge(tokens, ratio):
    # Simplified ToMe: match each token of set A to its most similar token of set B (cosine), merge the r best pairs by mean.
    r = merge_pairs(tokens.shape[1] - 1, ratio)
    if r <= 0:
        return tokens
    dtype, t = tokens.dtype, tokens.float()
    cls_tok, patch = t[:, :1], t[:, 1:]
    a, b = patch[:, 0::2], patch[:, 1::2]
    scores = F.normalize(a, dim=-1) @ F.normalize(b, dim=-1).transpose(-1, -2)
    node_max, node_idx = scores.max(dim=-1)
    order = node_max.argsort(dim=-1, descending=True)
    src_idx, unm_idx = order[:, :r], order[:, r:]
    dst_idx = node_idx.gather(-1, src_idx)
    D = a.shape[-1]
    unm = a.gather(1, unm_idx.unsqueeze(-1).expand(-1, -1, D))
    src = a.gather(1, src_idx.unsqueeze(-1).expand(-1, -1, D))
    b = b.scatter_reduce(1, dst_idx.unsqueeze(-1).expand(-1, -1, D), src, reduce="mean", include_self=True)
    return torch.cat([cls_tok, unm, b], dim=1).to(dtype)

def make_head(arch, is_final, dim=192):
    a = ARCHS[arch]
    body = nn.Linear(dim, NUM_CLASSES) if a["arch"] == "linear" else \
        nn.Sequential(nn.Linear(dim, a["hidden"]), nn.GELU(), nn.Linear(a["hidden"], NUM_CLASSES))
    # Intermediate CLS tokens are un-normalised; the final one already passed the backbone's own norm.
    return nn.Sequential(nn.Identity() if is_final else nn.LayerNorm(dim), body)

# A schedule is a tuple of merges ((layer, ratio), ...). A slot is (merges applied before this point, layer or 'final').
def all_schedules():
    singles = [(l, r) for l in EXIT_POOL for r in RATIOS]
    out = [()] + [(s,) for s in singles]
    if MAX_MERGES >= 2:
        out += [(a, b) for a, b in itertools.product(singles, singles) if a[0] < b[0]]
    return out

def prefix_at(sched, layer):
    return tuple(m for m in sched if m[0] < layer)     # exit is checked before a merge at the same layer

def sched_str(s):
    return "_".join(f"{l}r{int(r * 100)}" for l, r in s) or "none"

def slot_name(prefix, layer):
    return f"{sched_str(prefix)}__{layer}"

def all_slots():
    slots = set()
    for s in all_schedules():
        slots.add((s, "final"))
        for l in EXIT_POOL:
            slots.add((prefix_at(s, l), l))
    return sorted(slots, key=lambda x: (len(x[0]), str(x[0]), str(x[1])))

def all_designs():
    out = []
    exit_sets = [()]
    for k in range(1, MAX_EXITS + 1):
        for ls in itertools.combinations(EXIT_POOL, k):
            for arch in itertools.product(ARCHS, repeat=k):
                exit_sets.append(tuple(zip(ls, arch)))
    for s in all_schedules():
        for e in exit_sets:
            out.append({"merges": s, "exits": e, "final": "lin"})
    # Baselines with stronger final heads, so head capacity is not mistaken for an exit benefit.
    out += [{"merges": (), "exits": (), "final": a} for a in ARCHS if a != "lin"]
    return out

def design_id(d):
    e = "_".join(f"{l}{a}" for l, a in d["exits"]) or "none"
    return f"m[{sched_str(d['merges'])}]e[{e}]f[{d['final']}]"

def family(d):
    if d["merges"] and d["exits"]: return "exit+merge"
    if d["merges"]: return "merge_only"
    if d["exits"]: return "exit_only"
    return "baseline"

## 3. Energy meters
Preference order: GPU energy counter, then sampled GPU power, then wall clock (unit `s`, never mixed with joules). CPU uses RAPL.

In [ ]:
class Meter:
    kind, unit = "wallclock", "s"
    def read(self):
        return time.perf_counter()
    def close(self):
        pass

class NvmlCounter(Meter):
    kind, unit = "nvml_counter", "J"
    def __init__(self, nv, h):
        self.nv, self.h = nv, h
        nv.nvmlDeviceGetTotalEnergyConsumption(h)
    def read(self):
        return self.nv.nvmlDeviceGetTotalEnergyConsumption(self.h) / 1000.0

class NvmlPowerSampler(Meter):
    # For GPUs without the counter (common on laptops): integrates sampled power. Coarser, so it is labelled separately.
    kind, unit = "nvml_power_sampled", "J"
    def __init__(self, nv, h, period=0.02):
        self.nv, self.h, self.period = nv, h, period
        self.joules, self.last, self.stop = 0.0, time.perf_counter(), False
        self.watts = nv.nvmlDeviceGetPowerUsage(h) / 1000.0
        self.lock = threading.Lock()
        self.thread = threading.Thread(target=self._loop, daemon=True); self.thread.start()
    def _step(self):
        now = time.perf_counter()
        w = self.nv.nvmlDeviceGetPowerUsage(self.h) / 1000.0
        with self.lock:
            self.joules += 0.5 * (w + self.watts) * (now - self.last)
            self.watts, self.last = w, now
    def _loop(self):
        while not self.stop:
            time.sleep(self.period); self._step()
    def read(self):
        self._step()
        with self.lock:
            return self.joules
    def close(self):
        self.stop = True

class Rapl(Meter):
    # CPU package energy. Needs read access: sudo chmod a+r /sys/class/powercap/intel-rapl:0/energy_uj
    kind, unit = "rapl_package", "J"
    def __init__(self, root=None):
        root = root or os.environ.get("GV_RAPL_ROOT", "/sys/class/powercap/intel-rapl:0")
        self.path = os.path.join(root, "energy_uj")
        self.range = int(open(os.path.join(root, "max_energy_range_uj")).read())
        self.prev, self.total = int(open(self.path).read()), 0
    def read(self):
        now = int(open(self.path).read())
        self.total += (now - self.prev) if now >= self.prev else (now + self.range - self.prev)   # counter wraps
        self.prev = now
        return self.total / 1e6

def make_rapl():
    try:
        return Rapl()
    except Exception:
        return None

def make_meter(dev):
    if dev == "cpu":
        return make_rapl() or Meter()
    try:
        import pynvml
        pynvml.nvmlInit()
        h = pynvml.nvmlDeviceGetHandleByIndex(0)
    except Exception as e:
        print("NVML unavailable:", e)
        return Meter()
    for cls in (NvmlCounter, NvmlPowerSampler):
        try:
            return cls(pynvml, h)
        except Exception as e:
            print(cls.kind, "unsupported:", e)
    return Meter()

def gpu_state():
    # Temperature, clock and throttling change energy per operation, so they are recorded with every measurement.
    import subprocess
    q = "temperature.gpu,clocks.current.graphics,power.draw,clocks_event_reasons.sw_thermal_slowdown,clocks_event_reasons.hw_thermal_slowdown"
    try:
        v = subprocess.run(["nvidia-smi", f"--query-gpu={q}", "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=10).stdout.strip().split(", ")
        return {"temp_c": float(v[0]), "clock_mhz": float(v[1]), "power_w": float(v[2]), "sw_thermal_slowdown": v[3] == "Active", "hw_thermal_slowdown": v[4] == "Active"}
    except Exception:
        return {}

def wait_cool(dev, limit_c=COOL_TO_C, timeout_s=COOL_TIMEOUT_S):
    # Start every measurement block from a similar thermal state. Times out rather than blocking forever on a hot machine.
    if dev != "cuda":
        return
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        st = gpu_state()
        if not st or st["temp_c"] <= limit_c:
            print(f"GPU ready: {st}"); return
        time.sleep(15)
    print(f"WARNING: GPU still at {gpu_state().get('temp_c')} C after {timeout_s}s; measuring anyway, state is recorded.")

def hw_tag(dev):
    if HW_NAME:
        base = HW_NAME
    elif dev == "cuda":
        base = torch.cuda.get_device_name(0)
    else:
        m = re.search(r"model name\s*:\s*(.+)", open("/proc/cpuinfo").read()) if os.path.exists("/proc/cpuinfo") else None
        base = m.group(1) if m else (platform.processor() or "cpu")
    return dev + "_" + re.sub(r"[^a-z0-9]+", "-", base.lower()).strip("-")

class phase:
    # Logs energy and time of a pipeline stage, so the study's own footprint is reportable.
    def __init__(self, name, dev):
        self.name, self.dev = name, dev
    def __enter__(self):
        self.m = make_meter(self.dev); sync(self.dev)
        self.e0, self.t0 = self.m.read(), time.time()
    def __exit__(self, exc_type, *a):
        if exc_type is None:
            sync(self.dev)
            rec = load_json("phase_costs.json", {})
            rec.setdefault(hw_tag(self.dev), {})[self.name] = {"seconds": round(time.time() - self.t0, 1), "meter": self.m.kind,
                                                               "unit": self.m.unit, "value": round(self.m.read() - self.e0, 2)}
            save_json(rec, "phase_costs.json")
        self.m.close()

## 4. Stage `extract`: one tree-shaped pass caches CLS features for every merge schedule

In [ ]:
SLOTS = all_slots()
SPLITS_PATH = os.path.join(RESULTS, "splits.json")

def get_splits(Ytr):
    if os.path.exists(SPLITS_PATH):
        return json.load(open(SPLITS_PATH))
    y, used = Ytr.numpy(), []
    tr = stratified(y, TRAIN_PC, SEED); used += tr
    cal = stratified(y, CAL_PC, SEED + 1, used); used += cal
    val = stratified(y, VAL_PC, SEED + 2, used)
    sp = {"train": tr, "cal": cal, "val": val}
    json.dump(sp, open(SPLITS_PATH, "w"))
    return sp

def load_data():
    Xtr, Ytr, Xte, Yte = load_cifar()
    sp = get_splits(Ytr)
    te = list(range(min(TEST_N, len(Yte))))
    X = torch.cat([Xtr[sp["train"] + sp["cal"] + sp["val"]], Xte[te]])
    Y = torch.cat([Ytr[sp["train"] + sp["cal"] + sp["val"]], Yte[te]]).numpy()
    n = [len(sp["train"]), len(sp["cal"]), len(sp["val"]), len(te)]
    bounds = np.cumsum([0] + n)
    ranges = {k: (int(bounds[i]), int(bounds[i + 1])) for i, k in enumerate(["train", "cal", "val", "test"])}
    return X, Y, ranges

@torch.no_grad()
def walk(bb, tokens, start, prefix, sink):
    # Depth-first over merge schedules: shares every block before a branch point.
    for l in range(start + 1, len(bb.blocks) + 1):
        tokens = bb.blocks[l - 1](tokens)
        if l in EXIT_POOL:
            sink(prefix, l, tokens[:, 0])
            if len(prefix) < MAX_MERGES:
                for r in RATIOS:
                    walk(bb, token_merge(tokens, r), l, prefix + ((l, r),), sink)
    sink(prefix, "final", bb.norm(tokens)[:, 0])

def stage_extract():
    done_flag = os.path.join(FEATS, "DONE")
    if os.path.exists(done_flag):
        print("features already extracted"); return
    X, Y, ranges = load_data()
    np.save(os.path.join(FEATS, "labels.npy"), Y); save_json(ranges, "ranges.json")
    bb = make_backbone(DEV)
    n, D = len(X), bb.embed_dim
    print(f"{len(SLOTS)} feature slots, {n} images, {len(SLOTS) * n * D * 2 / 1e9:.2f} GB on disk")
    prog = os.path.join(FEATS, "progress.json")
    start = json.load(open(prog))["next"] if os.path.exists(prog) else 0
    mode = "r+" if start else "w+"
    mm = {}
    for s in SLOTS:
        path = os.path.join(FEATS, slot_name(*s) + ".npy")
        mm[slot_name(*s)] = np.lib.format.open_memmap(path, mode=mode, dtype=np.float16, shape=(n, D))
    t0 = time.time()
    for i in range(start, n, EXTRACT_BATCH):
        j = min(i + EXTRACT_BATCH, n)
        out = {}
        with amp_ctx(DEV):
            walk(bb, embed(bb, prepare(X[i:j], DEV)), 0, (), lambda p, l, c: out.__setitem__(slot_name(p, l), c.float().cpu()))
        for k, v in out.items():
            mm[k][i:j] = v.numpy().astype(np.float16)
        if (i // EXTRACT_BATCH) % 25 == 0:
            for m in mm.values():
                m.flush()
            json.dump({"next": j}, open(prog, "w"))
            rate = (j - start) / max(time.time() - t0, 1e-6)
            print(f"  {j}/{n} images | {rate:.1f} img/s | about {(n - j) / max(rate, 1e-6) / 60:.0f} min left", flush=True)
    for m in mm.values():
        m.flush()
    open(done_flag, "w").write("ok")
    del bb

## 5. Stage `train`: every head, 5 seeds, trained together
Adam updates each parameter independently, so stacking seeds gives independent models at roughly the cost of one.

In [ ]:
def fit_heads(X, y, arch, is_final, seeds, dev):
    members = []
    for s in seeds:
        torch.manual_seed(1000 + s)
        members.append(make_head(arch, is_final).to(dev))
    params, _ = stack_module_state(members)
    params = {k: v.detach().clone().requires_grad_(True) for k, v in params.items()}
    base = copy.deepcopy(members[0]).to("meta")
    vf = vmap(lambda p, x: functional_call(base, p, (x,)))
    opt = torch.optim.AdamW(params.values(), lr=LR, weight_decay=1e-4)
    n, M = len(X), len(seeds)
    steps = EPOCHS * math.ceil(n / BATCH)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.1)
    gens = [torch.Generator().manual_seed(2000 + s) for s in seeds]
    for _ in range(EPOCHS):
        perm = torch.stack([torch.randperm(n, generator=g) for g in gens]).to(dev)
        for i in range(0, n, BATCH):
            idx = perm[:, i:i + BATCH]
            logits = vf(params, X[idx])
            loss = F.cross_entropy(logits.flatten(0, 1), y[idx].flatten(), reduction="sum") / idx.shape[1]
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
    return {k: v.detach() for k, v in params.items()}, vf

def stacked_logits(vf, params, X, chunk=2048):
    M = next(iter(params.values())).shape[0]
    with torch.no_grad():
        return torch.cat([vf(params, X[i:i + chunk].unsqueeze(0).expand(M, -1, -1)) for i in range(0, len(X), chunk)], dim=1)

GRID_T = torch.logspace(-1.3, 1.0, 200)
def fit_temperature(logits, labels):
    nll = torch.stack([F.cross_entropy(logits / t, labels) for t in GRID_T])
    return float(GRID_T[nll.argmin()])

def ece(conf, correct, bins=15):
    edges, out = np.linspace(0, 1, bins + 1), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            out += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(out)

def signals(logits, T):
    p = F.softmax(logits / T, dim=-1)
    ent = -(p * p.clamp_min(1e-12).log()).sum(-1) / math.log(logits.shape[-1])
    conf, pred = p.max(-1)
    return ent.numpy().astype(np.float32), pred.numpy().astype(np.int8), conf.numpy().astype(np.float32)

def stage_train():
    assert os.path.exists(os.path.join(FEATS, "DONE")), "run stage 'extract' first"
    Y = torch.from_numpy(np.load(os.path.join(FEATS, "labels.npy")))
    R = load_json("ranges.json")
    seeds = list(range(N_SEEDS))
    rows, heads_dir = [], os.path.join(SCRATCH, "heads")
    os.makedirs(heads_dir, exist_ok=True)
    jobs = [(s, a) for s in SLOTS for a in ARCHS]
    y_dev = Y.to(DEV)
    t0 = time.time()
    for k, (slot, arch) in enumerate(jobs):
        key = f"{slot_name(*slot)}__{arch}"
        sig_path = os.path.join(SIGS, key + ".npz")
        csv_path = os.path.join(SIGS, key + ".json")
        if os.path.exists(sig_path) and os.path.exists(csv_path):
            rows += json.load(open(csv_path)); continue
        feats = torch.from_numpy(np.load(os.path.join(FEATS, slot_name(*slot) + ".npy")).astype(np.float32)).to(DEV)
        tr = R["train"]
        params, vf = fit_heads(feats[tr[0]:tr[1]], y_dev[tr[0]:tr[1]], arch, slot[1] == "final", seeds, DEV)
        logits = {}
        for sp in ("cal", "val", "test"):
            a, b = R[sp]
            logits[sp] = stacked_logits(vf, params, feats[a:b]).float().cpu()
        arrs, recs = {}, []
        for m in seeds:
            T = fit_temperature(logits["cal"][m], Y[R["cal"][0]:R["cal"][1]])
            rec = {"prefix": sched_str(slot[0]), "layer": str(slot[1]), "arch": arch, "seed": m, "temperature": T}
            for sp in ("val", "test"):
                ent, pred, conf = signals(logits[sp][m], T)
                arrs.setdefault(sp + "_ent", []).append(ent); arrs.setdefault(sp + "_pred", []).append(pred); arrs.setdefault(sp + "_conf", []).append(conf)
                yy = Y[R[sp][0]:R[sp][1]].numpy()
                raw = F.softmax(logits[sp][m], -1).max(-1).values.numpy()
                rec[f"{sp}_acc"] = float((pred == yy).mean())
                rec[f"{sp}_ece_before"] = ece(raw, pred == yy)
                rec[f"{sp}_ece_after"] = ece(conf, pred == yy)
            recs.append(rec)
        np.savez(sig_path, temps=np.array([r["temperature"] for r in recs]), **{k2: np.stack(v) for k2, v in arrs.items()})
        torch.save({"params": {k2: v.cpu().half() for k2, v in params.items()}}, os.path.join(heads_dir, key + ".pt"))
        json.dump(recs, open(csv_path, "w"))
        rows += recs
        if k % 20 == 0:
            rate = (k + 1) / max(time.time() - t0, 1e-6)
            print(f"  head group {k + 1}/{len(jobs)} | about {(len(jobs) - k) / rate / 60:.0f} min left (faster if resumed)", flush=True)
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(RESULTS, "slot_metrics.csv"), index=False)
    b0 = df[(df.prefix == "none") & (df.layer == "final") & (df.arch == "lin")].val_acc
    print(f"B0 validation accuracy: {b0.mean():.4f} +/- {b0.std():.4f}")
    if TRAIN_PC >= 1000 and not SMOKE and b0.mean() < B0_MIN_VAL_ACC:
        raise RuntimeError(f"B0 {b0.mean():.4f} is below {B0_MIN_VAL_ACC}; about 0.93 is expected. Fix this before going on.")

## 6. Stage `points`: evaluate every design offline over all thresholds and seeds

In [ ]:
def load_sigs():
    sig = {}
    for slot in SLOTS:
        for arch in ARCHS:
            key = f"{slot_name(*slot)}__{arch}"
            z = np.load(os.path.join(SIGS, key + ".npz"))
            sig[key] = {k: z[k] for k in z.files}
    return sig

def eval_design(d, sig, sp, y, seed, taus):
    # Exit at the first head whose calibrated normalised entropy is below tau. Returns preds, used-exit index.
    fin = sig[f"{slot_name(d['merges'], 'final')}__{d['final']}"]
    keys = [f"{slot_name(prefix_at(d['merges'], l), l)}__{a}" for l, a in d["exits"]]
    ents = [sig[k][sp + "_ent"][seed] for k in keys]
    preds = [sig[k][sp + "_pred"][seed] for k in keys]
    n, out_pred, out_used = len(y), [], []
    for t in taus:
        pred, used = fin[sp + "_pred"][seed].copy(), np.full(n, len(keys), dtype=np.int8)
        active = np.ones(n, dtype=bool)
        for i, (e, p) in enumerate(zip(ents, preds)):
            m = active & (e < t)
            pred[m], used[m] = p[m], i
            active &= ~m
        out_pred.append(pred); out_used.append(used)
    return np.stack(out_pred), np.stack(out_used)

def stage_points():
    sig = load_sigs()
    Y = np.load(os.path.join(FEATS, "labels.npy")); R = load_json("ranges.json")
    designs = all_designs()
    print(len(designs), "designs x", len(TAUS), "thresholds x", N_SEEDS, "seeds")
    T, S = len(TAUS), N_SEEDS
    res = {sp: {"acc": np.empty((len(designs), T, S), np.float32), "f1": np.empty((len(designs), T, S), np.float32),
                "f": np.empty((len(designs), T, S, MAX_EXITS), np.float32)} for sp in ("val", "test")}
    t0 = time.time()
    for di, d in enumerate(designs):
        for sp in ("val", "test"):
            y = Y[R[sp][0]:R[sp][1]]
            for s in range(S):
                pred, used = eval_design(d, sig, sp, y, s, TAUS)
                res[sp]["acc"][di, :, s] = (pred == y).mean(1)
                res[sp]["f1"][di, :, s] = macro_f1_batch(pred, y)
                for e in range(MAX_EXITS):
                    res[sp]["f"][di, :, s, e] = (used == e).mean(1) if e < len(d["exits"]) else 0.0
        if di % 500 == 0:
            print(f"  {di}/{len(designs)} | {time.time() - t0:.0f}s", flush=True)
    np.savez_compressed(os.path.join(RESULTS, "points.npz"),
                        **{f"{sp}_{k}": v for sp, dct in res.items() for k, v in dct.items()})
    json.dump([{"merges": d["merges"], "exits": d["exits"], "final": d["final"], "id": design_id(d)} for d in designs],
              open(os.path.join(RESULTS, "designs.json"), "w"))

def macro_f1_batch(pred, y):
    # pred: [T, N]
    out = np.zeros(pred.shape[0], np.float32)
    for c in range(NUM_CLASSES):
        p, t = pred == c, y == c
        tp = (p & t).sum(1); fp = (p & ~t).sum(1); fn = (~p & t).sum(1)
        out += np.where(tp > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)
    return out / NUM_CLASSES

## 7. Stage `energy`: measure a cost table for this machine
Every operation at every token count and batch size. Stores gross energy, energy net of idle, and time.

In [ ]:
def reachable_tokens(Np=NP):
    states, pairs = {1 + Np}, set()
    frontier = {1 + Np}
    for _ in range(MAX_MERGES):
        new = set()
        for N in frontier:
            for r in RATIOS:
                pairs.add((N, r)); new.add(1 + merged_count(N - 1, r))
        states |= new; frontier = new
    return sorted(states), sorted(pairs)

def bench(fn, meter, dev, idle_w, window=None, repeats=None):
    window, repeats = window or WINDOW_S, repeats or REPEATS
    for _ in range(5):
        fn()
    sync(dev); t0 = time.perf_counter()
    for _ in range(5):
        fn()
    sync(dev); per = (time.perf_counter() - t0) / 5
    calls = max(3, int(window / max(per, 1e-7)))
    gross, secs = [], []
    for _ in range(repeats):
        sync(dev); e0, t0 = meter.read(), time.perf_counter()
        for _ in range(calls):
            fn()
        sync(dev); e1, t1 = meter.read(), time.perf_counter()
        gross.append((e1 - e0) / calls); secs.append((t1 - t0) / calls)
    g, s = float(np.median(gross)), float(np.median(secs))
    return {"gross": g, "net": g - idle_w * s if meter.unit == "J" else g, "sec": s}

def measure_idle(meter, dev, bb):
    # A short load first, so the GPU sits in its active power state rather than deep idle.
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=dev)
    with torch.no_grad():
        for _ in range(20):
            embed(bb, x)
    sync(dev); e0, t0 = meter.read(), time.perf_counter()
    time.sleep(IDLE_S)
    return (meter.read() - e0) / (time.perf_counter() - t0) if meter.unit == "J" else 0.0

def regimes_here():
    out = [("cuda", b) for b in GPU_BATCHES] if DEV == "cuda" else []
    rapl = make_rapl() is not None
    if RUN_CPU_REGIME == "1" or (RUN_CPU_REGIME == "auto" and (rapl or DEV == "cpu")):
        out += [("cpu", b) for b in CPU_BATCHES]
    return out

def stage_energy():
    tables = load_json("energy_tables.json", {})
    for dev in sorted({d for d, _ in regimes_here()}):
        tag = hw_tag(dev)
        if tag in tables and tables[tag].get("complete"):
            print("cost table exists:", tag); continue
        wait_cool(dev)
        meter = make_meter(dev); bb = make_backbone(dev); D = bb.embed_dim
        state_start = gpu_state() if dev == "cuda" else {}
        states_log = []
        if meter.unit != "J":
            print(f"WARNING {tag}: no energy meter, recording wall-clock seconds only.")
        idle0 = measure_idle(meter, dev, bb)
        states, pairs = reachable_tokens(bb.patch_embed.num_patches)
        ops, blk = {}, bb.blocks[0]
        t0 = time.time()
        with torch.no_grad(), amp_ctx(dev):
            for b in GRID[dev]:
                print(f"  {tag}: batch {b} ({time.time() - t0:.0f}s elapsed)", flush=True)
                if dev == "cuda":
                    states_log.append({"batch": b, **gpu_state()})
                for N in states:
                    x = torch.randn(b, N, D, device=dev)
                    ops.setdefault(f"block|{N}", {})[b] = bench(lambda: blk(x), meter, dev, idle0)
                for N, r in pairs:
                    x = torch.randn(b, N, D, device=dev)
                    ops.setdefault(f"merge|{N}|{r}", {})[b] = bench(lambda: token_merge(x, r), meter, dev, idle0)
                img = torch.randn(b, 3, IMG_SIZE, IMG_SIZE, device=dev)
                ops.setdefault("embed", {})[b] = bench(lambda: embed(bb, img), meter, dev, idle0)
                cls = torch.randn(b, D, device=dev)
                for a in ARCHS:
                    head = make_head(a, False).to(dev).eval()
                    def head_fn():
                        p = F.softmax(head(cls).float(), -1)
                        (-(p * p.clamp_min(1e-12).log()).sum(-1) < 0.5).any().item()    # includes the exit-decision sync
                    ops.setdefault(f"head|{a}", {})[b] = bench(head_fn, meter, dev, idle0)
                    fh = make_head(a, True).to(dev).eval()
                    tok = torch.randn(b, 1 + NP, D, device=dev)
                    ops.setdefault(f"final|{a}", {})[b] = bench(lambda: fh(bb.norm(tok)[:, 0]).argmax(-1).sum().item(), meter, dev, idle0)
        idle1 = measure_idle(meter, dev, bb)
        tables = load_json("energy_tables.json", {})
        tables[tag] = {"device": dev, "meter": meter.kind, "unit": meter.unit, "idle_w_start": idle0, "idle_w_end": idle1, "gpu_state_start": state_start, "gpu_states": states_log, "gpu_state_end": gpu_state() if dev == "cuda" else {},
                       "batches": GRID[dev], "ops": {k: {str(b): v for b, v in d.items()} for k, d in ops.items()}, "complete": True}
        save_json(tables, "energy_tables.json")
        meter.close(); del bb

## 8. Cost models and stage `select`

In [ ]:
def design_ops(d, Np=NP):
    # Ordered operations of a design; `stage` = how many exits have already been passed (selects the survival fraction).
    N, stage, out = 1 + Np, 0, [("embed", "embed", 0)]
    exits, merges = dict(d["exits"]), dict(d["merges"])
    for l in range(1, 13):
        out.append(("block", N, stage))
        if l in exits:
            out.append(("head", exits[l], stage)); stage += 1
        if l in merges:
            out.append(("merge", (N, merges[l]), stage))
            N = 1 + merged_count(N - 1, merges[l])
    out.append(("final", d["final"], stage))
    return out

def analytic_op(kind, arg, D=192, C=NUM_CLASSES, Np=NP):
    if kind == "embed": return Np * 768 * D, 0.0
    if kind == "block": return 12 * arg * D * D + 2 * arg * arg * D, float(arg)
    if kind in ("head", "final"):
        a = ARCHS[arg]
        return (D * C if a["arch"] == "linear" else D * a["hidden"] + a["hidden"] * C), 0.0
    if kind == "merge":
        n = arg[0] - 1
        return ((n + 1) // 2) * (n // 2) * D, 0.0
    raise ValueError(kind)

def op_key(kind, arg):
    if kind == "embed": return "embed"
    if kind == "block": return f"block|{arg}"
    if kind == "merge": return f"merge|{arg[0]}|{arg[1]}"
    return f"{kind}|{arg}"

def nearest_block(ops, key):
    if key in ops:
        return ops[key]
    n = int(key.split("|")[1])
    best = min((int(k.split("|")[1]) for k in ops if k.startswith("block|")), key=lambda x: abs(x - n))
    return ops[f"block|{best}"]

_YS = {}
def interp_cost(op_tab, metric, batches, b):
    # Cost per call at (possibly fractional) batch b; below one sample, scale the single-sample cost by the probability of running.
    xs = np.log(np.array(batches, float))
    k = (id(op_tab), metric)
    if k not in _YS:
        _YS[k] = np.array([op_tab[str(x)][metric] for x in batches])
    ys = _YS[k]
    vals = np.interp(np.log(np.maximum(b, 1.0)), xs, ys)
    return np.where(b >= 1.0, vals, b * ys[0])

def survival(frac):
    # frac: [..., MAX_EXITS] -> list of survival arrays for stage 0..MAX_EXITS
    s = [np.ones(frac.shape[:-1])]
    for e in range(frac.shape[-1]):
        s.append(s[-1] - frac[..., e])
    return s

def merge_exec_frac(d, frac):
    # Fraction of samples that actually execute the first merge. Near zero means the merge is dead code in this design.
    if not d["merges"]:
        return np.zeros(frac.shape[:-1])
    s = survival(frac)
    return next(s[stage] for kind, _, stage in design_ops(d) if kind == "merge")

def analytic_costs(d, frac):
    s = survival(frac); mac = np.zeros_like(s[0]); tok = np.zeros_like(s[0])
    for kind, arg, stage in design_ops(d):
        m, t = analytic_op(kind, arg)
        mac += m * s[stage]; tok += t * s[stage]
    return mac, tok

def energy_cost(d, frac, table, batch, metric):
    # Shrinking batches: samples that exit leave the batch, so later operations run at a smaller effective batch size.
    s = survival(frac); total = np.zeros_like(s[0])
    for kind, arg, stage in design_ops(d):
        key = op_key(kind, arg)
        tab = nearest_block(table["ops"], key) if kind == "block" else table["ops"][key]
        total += interp_cost(tab, metric, table["batches"], batch * s[stage])
    return total / batch

def regime_list(tables):
    out = []
    for tag, t in tables.items():
        for b in (GPU_BATCHES if t["device"] == "cuda" else CPU_BATCHES):
            out.append((tag, b))
    return out

def load_designs():
    raw = json.load(open(os.path.join(RESULTS, "designs.json")))
    return [{"merges": tuple((l, r) for l, r in x["merges"]), "exits": tuple((l, a) for l, a in x["exits"]), "final": x["final"], "id": x["id"]} for x in raw]

def build_table():
    # One row per (design, threshold): every metric for every method, mean and spread across seeds.
    with np.load(os.path.join(RESULTS, "points.npz")) as z:
        P = {k: z[k] for k in z.files}      # an NpzFile decompresses on every access, so read each array once
    designs = load_designs()
    tables = load_json("energy_tables.json", {})
    regs = regime_list(tables)
    rows = []
    for di, d in enumerate(designs):
        frac = P["val_f"][di].astype(np.float64)         # [T, S, MAX_EXITS]
        frac_te = P["test_f"][di].astype(np.float64)
        mac, tok = analytic_costs(d, frac); mac_te, tok_te = analytic_costs(d, frac_te)
        mexec = merge_exec_frac(d, frac)
        costs = {}
        for tag, b in regs:
            for metric in ("gross", "net", "sec"):
                costs[f"{metric}__{tag}__b{b}"] = energy_cost(d, frac, tables[tag], b, metric).mean(1)
                costs[f"test_{metric}__{tag}__b{b}"] = energy_cost(d, frac_te, tables[tag], b, metric).mean(1)
        n_tau = 1 if not d["exits"] else len(TAUS)      # without exits every threshold is the same design
        for t in range(n_tau):
            r = {"design": d["id"], "family": family(d), "n_merges": len(d["merges"]), "n_exits": len(d["exits"]), "final_head": d["final"],
                 "merge_schedule": sched_str(d["merges"]),
                 "exits": "_".join(f"{l}{a}" for l, a in d["exits"]) or "none", "tau": TAUS[t],
                 "val_acc": P["val_acc"][di, t].mean(), "val_acc_std": P["val_acc"][di, t].std(ddof=1) if N_SEEDS > 1 else 0.0,
                 "test_acc": P["test_acc"][di, t].mean(), "test_acc_std": P["test_acc"][di, t].std(ddof=1) if N_SEEDS > 1 else 0.0,
                 "val_f1": P["val_f1"][di, t].mean(), "test_f1": P["test_f1"][di, t].mean(),
                 "macs": mac[t].mean(), "token_layers": tok[t].mean(), "test_macs": mac_te[t].mean(), "test_token_layers": tok_te[t].mean(),
                 "merge_exec_frac": mexec[t].mean(), "exit1_frac": frac[t, :, 0].mean(), "exit2_frac": frac[t, :, 1].mean() if MAX_EXITS > 1 else 0.0}
            for k, v in costs.items():
                r[k] = v[t]
            rows.append(r)
    df = pd.DataFrame(rows)
    # Keep these rows in the data, but do not let merging take credit for a merge almost nothing reaches.
    df.loc[(df.family == "exit+merge") & (df.merge_exec_frac < MIN_MERGE_USE), "family"] = "exit+merge_inactive"
    return df, regs, tables

def cost_signals(regs):
    # Signals a designer could optimise. Energy uses gross (what is actually drawn); net and time are stored too.
    return ["macs", "token_layers"] + [f"gross__{t}__b{b}" for t, b in regs]

def stage_select():
    df, regs, tables = build_table()
    df.to_csv(os.path.join(RESULTS, "all_points.csv.gz"), index=False)
    print("all_points.csv.gz:", len(df), "rows,", df.design.nunique(), "designs")
    b0_id = design_id({"merges": (), "exits": (), "final": "lin"})
    b0_val = float(df[df.design == b0_id].val_acc.iloc[0])
    print(f"B0 validation accuracy (mean across seeds): {b0_val:.4f}")
    picks = []
    for eps in EPS_LIST:
        feas = df[df.val_acc >= b0_val - eps]
        for fam in ("exit_only", "merge_only", "exit+merge"):
            f = feas[feas.family == fam]
            for sig in cost_signals(regs):
                if len(f) == 0:
                    picks.append({"eps": eps, "family": fam, "signal": sig, "design": None, "tau": None}); continue
                i = f[sig].idxmin()
                picks.append({"eps": eps, "family": fam, "signal": sig, "design": f.at[i, "design"], "tau": float(f.at[i, "tau"])})
        # best across all families, per signal: does the winner use merging?
        for sig in cost_signals(regs):
            i = feas[feas.family != "baseline"][sig].idxmin()
            picks.append({"eps": eps, "family": "any", "signal": sig, "design": feas.at[i, "design"], "tau": float(feas.at[i, "tau"])})
    picks = pd.DataFrame(picks); picks.to_csv(os.path.join(RESULTS, "picks.csv"), index=False)
    if os.path.exists(os.path.join(RESULTS, "finalists.json")) and "extract" not in STAGES and not os.path.isdir(SIGS):
        print("keeping existing finalists"); return
    fin = {(b0_id, 0.0)}
    for a in ("mlp128", "mlp192"):
        fin.add((design_id({"merges": (), "exits": (), "final": a}), 0.0))
    # Real runs are the costly part, so only the 0.5 and 1 point tolerances get them. All picks stay in picks.csv.
    fin |= {(r.design, r.tau) for r in picks.itertuples() if isinstance(r.design, str) and r.eps in FINALIST_EPS and r.signal != "token_layers"}
    fin = sorted(fin)
    print(len(fin), "finalist operating points")
    write_finalists(fin, df)

def paired_boot(a, b, n=2000, seed=0):
    d = a.astype(float) - b.astype(float); rng = np.random.RandomState(seed)
    m = np.array([d[rng.randint(0, len(d), len(d))].mean() for _ in range(n)])
    return float(d.mean()), float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))

def write_finalists(fin, df):
    sig = load_sigs(); Y = np.load(os.path.join(FEATS, "labels.npy")); R = load_json("ranges.json")
    y = Y[R["test"][0]:R["test"][1]]
    by_id = {d["id"]: d for d in load_designs()}
    real_idx = np.array(stratified(y, max(1, REAL_N // NUM_CLASSES), SEED + 9))
    np.random.RandomState(SEED + 10).shuffle(real_idx)      # stratified() is class-sorted; batches and time-boxed runs must be mixed
    b0 = by_id[design_id({"merges": (), "exits": (), "final": "lin"})]
    b0_ok = {s: eval_design(b0, sig, "test", y, s, [0.0])[0][0] == y for s in range(N_SEEDS)}
    rows, heads, recs = [], {}, []
    needed = set()
    for did, tau in fin:
        d = by_id[did]
        needed.add((slot_name(d["merges"], "final"), d["final"], True))
        for l, a in d["exits"]:
            needed.add((slot_name(prefix_at(d["merges"], l), l), a, False))
        row = {"design": did, "family": family(d), "tau": tau}
        accs, eces, diffs = [], [], []
        for s in range(N_SEEDS):
            pred, used = eval_design(d, sig, "test", y, s, [tau]); pred, used = pred[0], used[0]
            accs.append((pred == y).mean())
            conf = sig[f"{slot_name(d['merges'], 'final')}__{d['final']}"]["test_conf"][s].copy()
            for i, (l, a) in enumerate(d["exits"]):
                m = used == i
                conf[m] = sig[f"{slot_name(prefix_at(d['merges'], l), l)}__{a}"]["test_conf"][s][m]
            eces.append(ece(conf, pred == y))
            diffs.append(paired_boot(pred == y, b0_ok[s]))
        row.update(test_acc=float(np.mean(accs)), test_acc_std=float(np.std(accs, ddof=1)) if N_SEEDS > 1 else 0.0, test_ece=float(np.mean(eces)),
                   acc_diff_pp=100 * float(np.mean([x[0] for x in diffs])), acc_diff_seed_std_pp=100 * float(np.std([x[0] for x in diffs], ddof=1)) if N_SEEDS > 1 else 0.0,
                   diff_ci_lo_pp_seed0=100 * diffs[0][1], diff_ci_hi_pp_seed0=100 * diffs[0][2])
        pred0 = eval_design(d, sig, "test", y, 0, [tau])[0][0]
        row["offline_acc_real_subset_seed0"] = float((pred0[real_idx] == y[real_idx]).mean())
        rows.append(row)
        recs.append({"design": did, "tau": tau, "merges": d["merges"], "exits": d["exits"], "final": d["final"], "family": family(d)})
    pd.DataFrame(rows).to_csv(os.path.join(RESULTS, "finalist_metrics.csv"), index=False)
    # Seed-0 heads and temperatures for the real runs: small enough to download, unlike all 5 seeds of every head.
    for sl, a, final in sorted(needed):
        key = f"{sl}__{a}"
        ck = torch.load(os.path.join(SCRATCH, "heads", key + ".pt"), weights_only=True)
        heads[key] = {"state": {k: v[0].float() for k, v in ck["params"].items()}, "temp": float(sig[key]["temps"][0])}
    torch.save(heads, os.path.join(RESULTS, "finalist_heads.pt"))
    json.dump({"finalists": recs, "real_idx": [int(i) for i in real_idx]}, open(os.path.join(RESULTS, "finalists.json"), "w"), indent=1)

## 9. Stage `real`: run finalists for real and check the predictions
True early exit: samples leave the batch, later blocks never run for them. Designs are interleaved so drift hits all of them.

In [ ]:
class RealModel(nn.Module):
    def __init__(self, bb, rec, heads, dev):
        super().__init__()
        self.bb, self.merges, self.exits, self.final_key, self.dev = bb, {int(l): float(r) for l, r in rec["merges"]}, {}, None, dev
        sched = tuple((int(l), float(r)) for l, r in rec["merges"])
        def build(key, arch, final):
            h = make_head(arch, final); h.load_state_dict(heads[key]["state"]); return h.to(dev).eval(), heads[key]["temp"]
        for l, a in rec["exits"]:
            self.exits[int(l)] = build(f"{slot_name(prefix_at(sched, int(l)), int(l))}__{a}", a, False)
        self.final_head, _ = build(f"{slot_name(sched, 'final')}__{rec['final']}", rec["final"], True)

    @torch.no_grad()
    def forward(self, x, tau):
        # Returns predictions in input order, and the exit index of each sample.
        n = x.shape[0]; pred = torch.zeros(n, dtype=torch.long, device=x.device); alive = torch.arange(n, device=x.device)
        t = embed(self.bb, x)
        counts = {}
        for l, block in enumerate(self.bb.blocks, start=1):
            t = block(t)
            if l in self.exits:
                head, T = self.exits[l]
                lg = head(t[:, 0]).float()
                p = F.softmax(lg / T, -1)
                ent = -(p * p.clamp_min(1e-12).log()).sum(-1) / math.log(NUM_CLASSES)
                m = ent < tau
                k = int(m.sum().item())                # the exit decision forces a sync, as it would in deployment
                counts[l] = k
                if k:
                    pred[alive[m]] = lg[m].argmax(-1)
                    if k == alive.shape[0]:
                        return pred, counts
                    alive, t = alive[~m], t[~m]
            if l in self.merges:
                t = token_merge(t, self.merges[l])
        pred[alive] = self.final_head(self.bb.norm(t)[:, 0]).argmax(-1).long()
        return pred, counts

def real_once(model, tau, X, y, batch, dev, meter, aux, window):
    lat, done = [], 0
    n = X.shape[0]
    first_pass = []                                         # accuracy counts each image once; wrapped repeats only add measurement time
    sync(dev); e0 = meter.read(); a0 = aux.read() if aux else None; t0 = time.perf_counter()
    i = 0
    with torch.no_grad(), amp_ctx(dev):
        while True:
            idx = torch.arange(i, i + batch) % n
            xb = X[idx.to(X.device)]
            t1 = time.perf_counter()
            pred, _ = model(xb, tau)
            sync(dev)
            lat.append((time.perf_counter() - t1) * 1000 / batch)
            if i < n:
                first_pass.append((pred.cpu() == y[idx])[:n - i])
            done += batch; i += batch
            if time.perf_counter() - t0 >= window and done >= n:
                break
            if time.perf_counter() - t0 >= 4 * window:      # keeps a slow CPU from stalling; a partial pass is still a stratified sample
                break
    sec = time.perf_counter() - t0
    e = meter.read() - e0
    return {"samples": done, "seconds": sec, "gross_per_sample": e / done, "sec_per_sample": sec / done,
            "acc": float(torch.cat(first_pass).float().mean()) if done >= n else float("nan"),   # NaN if time-boxed before one full pass
            "aux_cpu_j_per_sample": ((aux.read() - a0) / done) if aux else None,
            "lat_p50_ms": float(np.percentile(lat, 50)), "lat_p95_ms": float(np.percentile(lat, 95)), "lat_p99_ms": float(np.percentile(lat, 99))}

def stage_real():
    tables = load_json("energy_tables.json", {})
    F_ = load_json("finalists.json"); heads = torch.load(os.path.join(RESULTS, "finalist_heads.pt"), weights_only=True)
    recs, real_idx = F_["finalists"], F_["real_idx"]
    _, _, Xte, Yte = load_cifar()
    Xte, Yte = Xte[:min(TEST_N, len(Yte))], Yte[:min(TEST_N, len(Yte))]
    rng = np.random.RandomState(SEED)
    out_path = os.path.join(RESULTS, "real_runs.csv")
    old = pd.read_csv(out_path) if os.path.exists(out_path) else pd.DataFrame()
    rows = []
    for dev, batch in regimes_here():
        tag = hw_tag(dev)
        if len(old) and ((old.hw == tag) & (old.batch == batch)).any():
            print("already measured:", tag, batch); continue
        wait_cool(dev)
        meter = make_meter(dev); bb = make_backbone(dev)
        aux = make_rapl() if dev == "cuda" else None
        idle_w = tables.get(tag, {}).get("idle_w_start", 0.0)
        xs = torch.cat([prepare(Xte[real_idx[i:i + 64]], dev) for i in range(0, len(real_idx), 64)])
        ys = Yte[real_idx]
        models = {(r["design"], r["tau"]): (RealModel(bb, r, heads, dev), r) for r in recs}
        order0 = list(models)
        print(f"{tag} batch {batch}: {len(order0)} operating points x {REPEATS} repeats, at least {WINDOW_S}s each "
              f"(at least {len(order0) * REPEATS * WINDOW_S / 60:.0f} min)", flush=True)
        for rep in range(REPEATS):
            order = [order0[i] for i in rng.permutation(len(order0))]
            for key in order:
                m, r = models[key]
                res = real_once(m, r["tau"], xs, ys, batch, dev, meter, aux, WINDOW_S)
                if dev == "cuda":
                    res.update({"gpu_" + k: v for k, v in gpu_state().items()})
                net = res["gross_per_sample"] - idle_w * res["sec_per_sample"] if meter.unit == "J" else res["gross_per_sample"]
                rows.append({"hw": tag, "device": dev, "batch": batch, "meter": meter.kind, "unit": meter.unit, "rep": rep,
                             "design": key[0], "tau": key[1], "family": r["family"], "net_per_sample": net, **res})
            pd.concat([old, pd.DataFrame(rows)]).to_csv(out_path, index=False)
        meter.close(); del bb, models

## 10. Stage `report`: the tables and figures for the paper
Every comparison below is a file in `results/`, so the paper can quote it directly.

In [ ]:
COL = {"exit_only": "#2a78d6", "merge_only": "#eb6834", "exit+merge": "#1baf7a", "baseline": "#52514e"}
MRK = {"exit_only": "o", "merge_only": "s", "exit+merge": "^", "baseline": "D"}
LABEL = {"exit_only": "Exit only", "merge_only": "Merge only", "exit+merge": "Exit + merge", "baseline": "Baseline (B0)"}
INK, INK2, GRIDC = "#0b0b0b", "#52514e", "#e6e5e1"

def style(ax):
    ax.grid(True, color=GRIDC, lw=0.6); ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRIDC)
    ax.tick_params(colors=INK2, labelsize=8, length=0)
    ax.xaxis.label.set_color(INK2); ax.yaxis.label.set_color(INK2)
    ax.title.set_color(INK)

def front(df, cost):
    f, best = [], -1.0
    for r in df.sort_values([cost, "val_acc"], ascending=[True, False]).itertuples():
        a = r.val_acc
        if a > best + 1e-9:
            f.append(r.Index); best = a
    return df.loc[f]

def unit_of(tables, tag):
    return tables[tag]["unit"]

def dev_name(tag):
    # Label a cost table: which device, and for a GPU, which number format it ran in.
    if tag.startswith("cpu"):
        return "CPU"
    return "GPU float32" if tag.endswith("fp32") else "GPU float16"

def reg_name(tag, b):
    return f"{dev_name(tag)} batch {b}"

def stage_report():
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.size": 9, "figure.dpi": 150, "savefig.bbox": "tight", "axes.titlesize": 10, "axes.titleweight": "bold",
                         "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb"})
    df, regs, tables = build_table()
    df.to_csv(os.path.join(RESULTS, "all_points.csv.gz"), index=False)
    if not regs:
        print("no cost tables yet: run stage 'energy'"); return
    b0 = df[(df.family == "baseline") & (df.final_head == "lin")].iloc[0]
    REF = float(b0.val_acc)
    # The float32 GPU table was measured at the 92 C limit with clocks swinging, and overshoots real runs up to 2.9x.
    # Its rows stay in the CSVs, but figures that rest on the table leave it out. Its evidence is the cooled real runs.
    STABLE = [t for t in tables if not t.endswith("fp32")]
    regs_stable = [(t, b) for t, b in regs if t in STABLE]
    lines = ["# Results summary", "", f"B0 validation accuracy {REF:.4f}, test accuracy {b0.test_acc:.4f} (+/- {b0.test_acc_std:.4f}).", ""]

    # --- method comparison: best per family, per regime and tolerance; selected on val, reported on test ---
    rows = []
    for tag, b in regs:
        cost, net, sec = f"gross__{tag}__b{b}", f"net__{tag}__b{b}", f"sec__{tag}__b{b}"
        tc, tn, ts = "test_" + cost, "test_" + net, "test_" + sec
        for eps in EPS_LIST:
            for fam in ("exit_only", "merge_only", "exit+merge"):
                f = df[(df.family == fam) & (df.val_acc >= REF - eps)]
                base = {"hw": tag, "batch": b, "unit": unit_of(tables, tag), "meter": tables[tag]["meter"], "eps_pp": eps * 100, "family": fam}
                if not len(f):
                    rows.append({**base, "feasible": False}); continue
                r = f.loc[f[cost].idxmin()]
                rows.append({**base, "feasible": True, "design": r.design, "tau": r.tau, "test_acc": r.test_acc, "test_acc_std": r.test_acc_std,
                             "acc_diff_pp": 100 * (r.test_acc - b0.test_acc), "gross_saved_pct": 100 * (1 - r[tc] / b0[tc]),
                             "net_saved_pct": 100 * (1 - r[tn] / b0[tn]), "time_saved_pct": 100 * (1 - r[ts] / b0[ts]),
                             "macs_saved_pct": 100 * (1 - r.test_macs / b0.test_macs), "token_layers_saved_pct": 100 * (1 - r.test_token_layers / b0.test_token_layers)})
    cmp_df = pd.DataFrame(rows); cmp_df.to_csv(os.path.join(RESULTS, "method_comparison.csv"), index=False)

    # --- does merging add anything beyond exits? ---
    rows = []
    for (tag, b, eps), g in cmp_df[cmp_df.feasible].groupby(["hw", "batch", "eps_pp"]):
        e = g[g.family == "exit_only"]; m = g[g.family == "exit+merge"]; o = g[g.family == "merge_only"]
        rows.append({"hw": tag, "batch": b, "eps_pp": eps,
                     "exit_only_saved_pct": e.gross_saved_pct.max() if len(e) else np.nan,
                     "merge_only_saved_pct": o.gross_saved_pct.max() if len(o) else np.nan,
                     "exit_merge_saved_pct": m.gross_saved_pct.max() if len(m) else np.nan})
    add = pd.DataFrame(rows)
    if len(add):
        add["merge_adds_pp"] = add.exit_merge_saved_pct - add.exit_only_saved_pct
    add.to_csv(os.path.join(RESULTS, "merge_added_value.csv"), index=False)

    # --- break-even: blocks a merge must run before it repays itself ---
    rows = []
    for tag, t in tables.items():
        for bs in t["batches"]:
            for key, cost_tab in t["ops"].items():
                if not key.startswith("merge|"):
                    continue
                _, N, r = key.split("|"); N, r = int(N), float(r)
                N2 = 1 + merged_count(N - 1, r)
                for metric in ("gross", "net", "sec"):
                    E_b0 = nearest_block(t["ops"], f"block|{N}")[str(bs)][metric]; E_b1 = nearest_block(t["ops"], f"block|{N2}")[str(bs)][metric]
                    saving, mc = E_b0 - E_b1, cost_tab[str(bs)][metric]
                    rows.append({"hw": tag, "batch": bs, "metric": metric, "unit": t["unit"], "tokens_before": N, "tokens_after": N2, "ratio": r,
                                 "merge_cost_per_batch": mc, "block_saving_per_batch": saving,
                                 "merge_cost_in_blocks": mc / max(E_b0, 1e-12), "block_saving_pct": 100 * saving / max(E_b0, 1e-12),
                                 "breakeven_blocks": mc / saving if saving > 0 else np.inf,
                                 "mac_breakeven_blocks": analytic_op("merge", (N, r))[0] / (analytic_op("block", N)[0] - analytic_op("block", N2)[0])})
    be = pd.DataFrame(rows); be.to_csv(os.path.join(RESULTS, "merge_breakeven.csv"), index=False)
    pd.DataFrame([{"hw": tag, "regime": dev_name(tag), "batch": int(bs), "op": k, "unit": t["unit"], **v} for tag, t in tables.items() for k, d in t["ops"].items() for bs, v in d.items()]
                 ).to_csv(os.path.join(RESULTS, "op_costs.csv"), index=False)

    # --- merge-only effect per schedule: what does merging cost in accuracy, what does it save? ---
    mo = df[df.family == "merge_only"].copy()
    keep = ["merge_schedule", "n_merges", "val_acc", "val_acc_std", "test_acc", "test_acc_std", "test_f1"]
    me = mo[keep].copy(); me["acc_diff_pp"] = 100 * (mo.test_acc - b0.test_acc); me["macs_saved_pct"] = 100 * (1 - mo.test_macs / b0.test_macs)
    me["token_layers_saved_pct"] = 100 * (1 - mo.test_token_layers / b0.test_token_layers)
    for tag, b in regs:
        for metric in ("gross", "net", "sec"):
            c = f"test_{metric}__{tag}__b{b}"
            me[f"{metric}_saved_pct__{tag}__b{b}"] = 100 * (1 - mo[c] / b0[c])
    me.sort_values("acc_diff_pp", ascending=False).to_csv(os.path.join(RESULTS, "merge_only_effect.csv"), index=False)

    # --- do MACs choose the right design? regret of the MAC pick, in measured-table energy ---
    rows = []
    for tag, b in regs:
        cost = f"gross__{tag}__b{b}"; tc = "test_" + cost; pool = df[df.family != "baseline"]
        rho = pool[[cost, "macs"]].corr("spearman").iloc[0, 1]
        for eps in EPS_LIST:
            f = pool[pool.val_acc >= REF - eps]
            if not len(f):
                continue
            me_i, mm_i, mt_i = f[cost].idxmin(), f["macs"].idxmin(), f["token_layers"].idxmin()
            e, mp = f.loc[me_i], f.loc[mm_i]
            rows.append({"hw": tag, "batch": b, "eps_pp": eps * 100, "spearman_energy_vs_macs": rho, "energy_pick": e.design, "energy_pick_tau": e.tau,
                         "energy_pick_family": e.family, "mac_pick": mp.design, "mac_pick_tau": mp.tau, "mac_pick_family": mp.family,
                         "same_design": e.design == mp.design and e.tau == mp.tau, "energy_pick_saved_pct": 100 * (1 - e[tc] / b0[tc]), "mac_pick_saved_pct": 100 * (1 - mp[tc] / b0[tc]),
                         "macs_claimed_saved_pct": 100 * (1 - mp.test_macs / b0.test_macs),
                         "mac_regret_pct": 100 * (mp[tc] / e[tc] - 1), "token_pick_regret_pct": 100 * (f.loc[mt_i, tc] / e[tc] - 1)})
    ag = pd.DataFrame(rows); ag.to_csv(os.path.join(RESULTS, "mac_vs_energy_agreement.csv"), index=False)

    # --- real measurements: aggregate, compare to the prediction ---
    real_path, rv = os.path.join(RESULTS, "real_runs.csv"), None
    if os.path.exists(real_path):
        raw = pd.read_csv(real_path)
        rv = raw.groupby(["hw", "device", "batch", "meter", "unit", "design", "tau", "family"], as_index=False).agg(
            gross=("gross_per_sample", "median"), gross_mean=("gross_per_sample", "mean"), gross_std=("gross_per_sample", "std"),
            net=("net_per_sample", "median"), sec=("sec_per_sample", "median"),
            lat_p50_ms=("lat_p50_ms", "mean"), lat_p95_ms=("lat_p95_ms", "mean"), lat_p99_ms=("lat_p99_ms", "mean"), real_acc=("acc", "mean"),
            aux_cpu_j=("aux_cpu_j_per_sample", "median"), samples=("samples", "mean"))
        look = df.set_index(["design", "tau"])
        preds = []
        for r in rv.itertuples():
            k = (r.design, r.tau)
            preds.append(look.at[k, f"test_gross__{r.hw}__b{r.batch}"] if k in look.index and f"test_gross__{r.hw}__b{r.batch}" in look.columns else np.nan)
        rv["pred_gross"] = np.array(preds, dtype=float)
        rv["pred_error_pct"] = 100 * (rv.pred_gross / rv.gross - 1)
        ref = rv[rv.design == b0.design].set_index(["hw", "batch"]).gross
        rv["measured_saved_pct"] = [100 * (1 - g / ref.get((h, b), np.nan)) for g, h, b in zip(rv.gross, rv.hw, rv.batch)]
        if rv.unit.iloc[0] == "J" or (rv.unit == "J").any():
            rv["co2_g_per_1k"] = np.where(rv.unit == "J", rv.gross / 3.6e6 * CARBON_KG_PER_KWH * 1e6, np.nan)
        fm_path = os.path.join(RESULTS, "finalist_metrics.csv")
        if os.path.exists(fm_path):
            fm = pd.read_csv(fm_path)[["design", "tau", "offline_acc_real_subset_seed0"]]
            rv = rv.merge(fm, on=["design", "tau"], how="left")
            rv["real_minus_offline_pp"] = 100 * (rv.real_acc - rv.offline_acc_real_subset_seed0)
        rv.to_csv(os.path.join(RESULTS, "real_measured.csv"), index=False)

        # --- the main comparison: each regime's validation pick, measured for real, against B0 in the same repeat ---
        fmx = pd.read_csv(fm_path).set_index(["design", "tau"]) if os.path.exists(fm_path) else None
        rows = []
        for c in cmp_df[cmp_df.feasible].itertuples():
            q = raw[(raw.hw == c.hw) & (raw.batch == c.batch)]
            a = q[(q.design == c.design) & (q.tau == c.tau)].set_index("rep"); z = q[q.design == b0.design].set_index("rep")
            row = {"hw": c.hw, "regime": reg_name(c.hw, c.batch), "batch": c.batch, "eps_pp": c.eps_pp, "family": c.family, "design": c.design, "tau": c.tau, "measured": len(a) > 0,
                   "test_acc": c.test_acc, "acc_diff_pp": c.acc_diff_pp, "predicted_saved_pct": c.gross_saved_pct, "macs_saved_pct": c.macs_saved_pct}
            if len(a):
                sv = 100 * (1 - a.gross_per_sample / z.gross_per_sample)
                row.update(measured_saved_pct=sv.median(), measured_saved_min=sv.min(), measured_saved_max=sv.max(),
                           net_saved_pct=(100 * (1 - a.net_per_sample / z.net_per_sample)).median(), time_saved_pct=(100 * (1 - a.sec_per_sample / z.sec_per_sample)).median(),
                           real_acc=a.acc.mean(), latency_p95_ms=a.lat_p95_ms.median(), mj_per_sample=1000 * a.gross_per_sample.median())
                if a.aux_cpu_j_per_sample.notna().all() and len(a):
                    row["with_cpu_saved_pct"] = (100 * (1 - (a.gross_per_sample + a.aux_cpu_j_per_sample) / (z.gross_per_sample + z.aux_cpu_j_per_sample))).median()
            if fmx is not None and (c.design, c.tau) in fmx.index:
                row.update(diff_ci_lo_pp_seed0=fmx.at[(c.design, c.tau), "diff_ci_lo_pp_seed0"], diff_ci_hi_pp_seed0=fmx.at[(c.design, c.tau), "diff_ci_hi_pp_seed0"])
            rows.append(row)
        mc_df = pd.DataFrame(rows); mc_df.to_csv(os.path.join(RESULTS, "measured_comparison.csv"), index=False)

    # --- figure 1: fronts per regime, one small multiple each ---
    n = len(regs_stable); cols = min(3, n); rws = math.ceil(n / cols)
    fig, axes = plt.subplots(rws, cols, figsize=(3.6 * cols, 2.9 * rws), squeeze=False)
    floor = REF - 0.05
    for ax, (tag, b) in zip(axes.flat, regs_stable):
        c = f"gross__{tag}__b{b}"; k = 1000 if unit_of(tables, tag) == "J" else 1000
        for fam in ("exit_only", "merge_only", "exit+merge"):
            f = front(df[(df.family == fam) & (df.val_acc >= floor)], c)
            if len(f):
                ax.plot(f[c] * k, f.val_acc * 100, color=COL[fam], lw=2, marker=MRK[fam], ms=4, label=LABEL[fam], mec="#fcfcfb", mew=0.6)
        ax.axhline((REF - 0.01) * 100, color=INK2, lw=0.8, ls=(0, (3, 3)))
        ax.scatter([b0[c] * k], [REF * 100], color=COL["baseline"], marker="D", s=30, zorder=5, label=LABEL["baseline"], ec="#fcfcfb", lw=0.6)
        ax.set_title(reg_name(tag, b)); style(ax)
        ax.set_xlabel(("mJ" if unit_of(tables, tag) == "J" else "ms (no energy meter)") + " per sample, predicted"); ax.set_ylabel("Validation accuracy (%)")
    for ax in list(axes.flat)[n:]:
        ax.axis("off")
    h, l = axes.flat[0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=4, frameon=False, fontsize=8, bbox_to_anchor=(0.5, -0.04))
    fig.suptitle("Accuracy against cost: dashed line is the 1-point tolerance", fontsize=10, color=INK, fontweight="bold")
    fig.tight_layout(); fig.savefig(os.path.join(RESULTS, "fig_fronts.png"), dpi=200); plt.close(fig)

    # --- figure 2: merge break-even heatmap (one sequential hue; never-pays cells hatched and labelled) ---
    sub = be[be.metric == "gross"]
    tags = STABLE
    fig, axes = plt.subplots(1, len(tags), figsize=(4.2 * len(tags), 3.0), squeeze=False)
    from matplotlib.colors import LinearSegmentedColormap
    cmap = LinearSegmentedColormap.from_list("seq", ["#e8f1fb", "#2a78d6", "#0d3a73"])
    for ax, tag in zip(axes.flat, tags):
        g = sub[sub.hw == tag].copy(); g["cell"] = g.tokens_before.astype(str) + " → " + g.tokens_after.astype(str)
        piv = g.pivot_table(index="cell", columns="batch", values="breakeven_blocks", aggfunc="first")
        vals = piv.values.astype(float); shown = np.where(vals <= 12, vals, np.nan)    # off-scale cells stay empty, hatched and labelled
        ax.imshow(shown, cmap=cmap, vmin=0, vmax=12, aspect="auto")
        for i in range(vals.shape[0]):
            for j in range(vals.shape[1]):
                v = vals[i, j]
                if not np.isfinite(v) or v > 12:
                    ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, fill=False, hatch="///", ec="#c9c7c1", lw=0))
                    ax.text(j, i, "never" if not np.isfinite(v) else ">12", ha="center", va="center", fontsize=7, color=INK, bbox=dict(fc="#fcfcfb", ec="none", pad=1.5))
                else:
                    ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=7, color="#fcfcfb" if v > 6 else INK)
        ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns); ax.set_yticks(range(len(piv.index))); ax.set_yticklabels(piv.index)
        ax.set_xlabel("Batch size"); ax.set_ylabel("Tokens (before → after merge)")
        ax.set_title(dev_name(tag)); ax.tick_params(length=0, labelsize=8, colors=INK2)
        for s in ax.spines.values():
            s.set_visible(False)
    word = "energy" if all(t["unit"] == "J" for t in tables.values()) else "cost (time where no energy meter exists)"
    fig.suptitle(f"Blocks needed for a merge to repay its {word}; at most 10 blocks remain after layer 2", fontsize=10, color=INK, fontweight="bold")
    fig.tight_layout(); fig.savefig(os.path.join(RESULTS, "fig_merge_breakeven.png"), dpi=200); plt.close(fig)

    # --- figure 3: MAC pick versus energy pick, 1-point tolerance ---
    a1 = ag[np.isclose(ag.eps_pp, 1.0) & ag.hw.isin(STABLE)]
    if len(a1):
        fig, ax = plt.subplots(figsize=(max(5, 1.3 * len(a1)), 3.0)); x = np.arange(len(a1)); w = 0.36
        ax.bar(x - w / 2 - .01, a1.energy_pick_saved_pct, w, color="#2a78d6", label="Chosen by energy")
        ax.bar(x + w / 2 + .01, a1.mac_pick_saved_pct, w, color="#eb6834", label="Chosen by MACs")
        ax.set_xticks(x); ax.set_xticklabels([reg_name(h, b).replace(" batch", "\nbatch") for h, b in zip(a1.hw, a1.batch)], fontsize=8)
        ax.set_ylabel("Cost saved vs B0 (%)"); ax.axhline(0, color=INK2, lw=0.8); style(ax); ax.grid(axis="x", visible=False)
        ax.legend(frameon=False, fontsize=8, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.12)); ax.set_title("Cost saved by the design each signal chooses (1-point tolerance)")
        fig.savefig(os.path.join(RESULTS, "fig_mac_vs_energy.png"), dpi=200); plt.close(fig)

    # --- figure 4: predicted against measured ---
    if rv is not None and rv.pred_gross.notna().any():
        groups = list(rv.groupby(["hw", "batch"]))
        cols = min(3, len(groups)); rws = math.ceil(len(groups) / cols)
        fig, axes = plt.subplots(rws, cols, figsize=(3.2 * cols, 3.0 * rws), squeeze=False)
        for ax, ((h, b), g) in zip(axes.flat, groups):
            k = 1000
            for fam, gg in g.groupby("family"):
                ax.scatter(gg.pred_gross * k, gg.gross * k, s=26, color=COL[fam], marker=MRK[fam], label=LABEL[fam], ec="#fcfcfb", lw=0.6)
            lim = [0, 1.05 * max(np.nanmax(g.pred_gross), g.gross.max()) * k]
            ax.plot(lim, lim, color=INK2, lw=0.8, ls=(0, (3, 3))); ax.set_xlim(lim); ax.set_ylim(lim)
            u_ = "mJ" if g.unit.iloc[0] == "J" else "ms"
            ax.set_title(reg_name(h, b)); style(ax); ax.set_xlabel(f"Predicted per sample ({u_})"); ax.set_ylabel(f"Measured per sample ({u_})")
        for ax in list(axes.flat)[len(groups):]:
            ax.axis("off")
        h_, l_ = axes.flat[0].get_legend_handles_labels()
        fig.legend(h_, l_, loc="lower center", ncol=4, frameon=False, fontsize=8, bbox_to_anchor=(0.5, -0.04))
        fig.suptitle("Is the cost table right? On the dashed line, prediction equals measurement", fontsize=10, color=INK, fontweight="bold")
        fig.tight_layout(); fig.savefig(os.path.join(RESULTS, "fig_pred_vs_measured.png"), dpi=200); plt.close(fig)

    # --- figure 5: accuracy cost of merging by when and how hard ---
    one = me[me.n_merges == 1].copy()
    if len(one):
        one["layer"] = one.merge_schedule.str.extract(r"^(\d+)r").astype(int); one["ratio"] = one.merge_schedule.str.extract(r"r(\d+)$").astype(int)
        fig, ax = plt.subplots(figsize=(4.8, 3.0))
        for rr, c, mk in ((25, "#2a78d6", "o"), (50, "#eb6834", "s")):
            g = one[one.ratio == rr].sort_values("layer")
            ax.errorbar(g.layer, g.acc_diff_pp, yerr=100 * g.test_acc_std, color=c, marker=mk, ms=5, lw=2, capsize=2, label=f"{rr}% of tokens removed")
        ax.axhline(0, color=INK2, lw=0.8); ax.axhline(-1, color=INK2, lw=0.8, ls=(0, (3, 3))); style(ax)
        ax.set_xlabel("Block where merging happens"); ax.set_ylabel("Test accuracy vs B0 (points)"); ax.set_xticks(EXIT_POOL); ax.set_title("Accuracy cost of one merge (dashed: 1-point tolerance)")
        ax.legend(frameon=False, fontsize=8); fig.savefig(os.path.join(RESULTS, "fig_merge_accuracy.png"), dpi=200); plt.close(fig)

    # --- figure 6: measured savings per regime and family, 1-point tolerance ---
    if rv is not None and len(mc_df):
        m1 = mc_df[np.isclose(mc_df.eps_pp, 1.0) & mc_df.measured & mc_df.hw.isin(STABLE)]
        cool_path = os.path.join(RESULTS, "real_runs_float32_cooled.csv")
        if os.path.exists(cool_path):
            cr = pd.read_csv(cool_path); rows = []
            for (bt, ds, tu, fam, kd), g in cr[cr.family != "baseline"].groupby(["batch", "design", "tau", "family", "kind"]):
                z = cr[(cr.batch == bt) & (cr.family == "baseline")].set_index("rep"); a = g.set_index("rep")
                sv = 100 * (1 - a.gross_per_sample / z.gross_per_sample.reindex(a.index))
                acc = df[(df.design == ds) & (df.tau == tu)].test_acc.iloc[0]
                rows.append({"regime": f"GPU float32 cooled batch {bt}", "batch": bt, "family": fam, "design": ds, "tau": tu, "kind": kd, "repeats": len(a), "acc_diff_pp": 100 * (acc - b0.test_acc),
                             "measured_saved_pct": sv.median(), "measured_saved_min": sv.min(), "measured_saved_max": sv.max(),
                             "time_saved_pct": (100 * (1 - a.sec_per_sample / z.sec_per_sample.reindex(a.index))).median(),
                             "with_cpu_saved_pct": (100 * (1 - (a.gross_per_sample + a.aux_cpu_j_per_sample) / (z.gross_per_sample + z.aux_cpu_j_per_sample).reindex(a.index))).median(),
                             "macs_saved_pct": 100 * (1 - df[(df.design == ds) & (df.tau == tu)].test_macs.iloc[0] / b0.test_macs),
                             "mj_per_sample": 1000 * a.gross_per_sample.median(), "b0_mj_per_sample": 1000 * z.gross_per_sample.median(), "real_acc": a.acc.mean(),
                             "max_end_temp_c": a.gpu_temp_c.max(), "slowdown_frac": a.gpu_sw_thermal_slowdown.astype(bool).mean()})
            cool = pd.DataFrame(rows); cool.to_csv(os.path.join(RESULTS, "float32_cooled_summary.csv"), index=False)
            # Batch 128 exit: use the design measured to match the other regimes; the throttled pick stays in the CSV.
            show = cool[~((cool.batch == 128) & (cool.family == "exit_only") & (cool.kind == "pick"))]
            m1 = pd.concat([m1, show.assign(predicted_saved_pct=np.nan)], ignore_index=True)
        names = list(dict.fromkeys(m1.regime)); fams = ("exit_only", "merge_only", "exit+merge"); w = 0.27
        fig, ax = plt.subplots(figsize=(max(6, 1.25 * len(names)), 3.4)); x = np.arange(len(names))
        for i, fam in enumerate(fams):
            g = m1[m1.family == fam].set_index("regime").reindex(names); xs = x + (i - 1) * w
            ax.bar(xs, g.measured_saved_pct, w - .02, color=COL[fam], label=LABEL[fam])
            ax.errorbar(xs, g.measured_saved_pct, yerr=[g.measured_saved_pct - g.measured_saved_min, g.measured_saved_max - g.measured_saved_pct], fmt="none", ecolor=INK, elinewidth=.8, capsize=2)
            ax.scatter(xs, g.predicted_saved_pct, marker="_", s=70, color=INK, zorder=5, label="Predicted by cost table" if i == 0 else None)
        ax.set_xticks(x); ax.set_xticklabels([n.replace(" cooled", "\ncooled").replace(" batch", "\nbatch") for n in names], fontsize=8); ax.set_ylabel("Energy saved vs B0 (%)"); style(ax); ax.grid(axis="x", visible=False)
        ax.legend(frameon=False, fontsize=8, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.16)); ax.set_title("Measured energy saved within 1 point of B0 (whiskers: range over repeats)")
        fig.savefig(os.path.join(RESULTS, "fig_measured_savings.png"), dpi=200); plt.close(fig)

    # --- summary ---
    bl = df[df.family == "baseline"][["final_head", "val_acc", "test_acc", "test_acc_std"]].round(4)
    lines += ["## Baselines by final head (a stronger head can beat exits without any exit benefit)", "",
              bl.to_markdown(index=False) if _has_tabulate() else bl.to_string(index=False), ""]
    lines += ["## Best per family (1-point tolerance; selected on validation, reported on test)", ""]
    c1 = cmp_df[np.isclose(cmp_df.eps_pp, 1.0)]
    cols_ = [c for c in ["hw", "batch", "family", "feasible", "test_acc", "acc_diff_pp", "gross_saved_pct", "net_saved_pct", "time_saved_pct", "macs_saved_pct", "design", "tau"] if c in c1]
    lines += [c1[cols_].round(3).to_markdown(index=False) if _has_tabulate() else c1[cols_].round(3).to_string(index=False), ""]
    if len(ag):
        lines += ["## Do MACs pick the right design? (1-point tolerance)", ""]
        a = ag[np.isclose(ag.eps_pp, 1.0)][["hw", "batch", "spearman_energy_vs_macs", "same_design", "energy_pick_family", "mac_pick_family", "energy_pick_saved_pct", "mac_pick_saved_pct", "mac_regret_pct"]].round(3)
        lines += [a.to_markdown(index=False) if _has_tabulate() else a.to_string(index=False), ""]
    pc = load_json("phase_costs.json", {})
    lines += ["## Footprint of this study", "", "```", json.dumps(pc, indent=1), "```", ""]
    if rv is not None:
        lines += [f"Real runs: {len(rv)} operating point x regime rows, median of repeats. Median absolute prediction error: {rv.pred_error_pct.abs().median():.1f}%. It differs by regime:", ""]
        pe = rv.assign(e=rv.pred_error_pct.abs()).groupby(["hw", "batch"]).agg(median_abs_err_pct=("e", "median"), max_abs_err_pct=("e", "max")).round(1).reset_index()
        lines += [pe.to_markdown(index=False) if _has_tabulate() else pe.to_string(index=False), ""]
        m1 = mc_df[np.isclose(mc_df.eps_pp, 1.0)][["regime", "family", "acc_diff_pp", "predicted_saved_pct", "measured_saved_pct", "measured_saved_min", "measured_saved_max", "time_saved_pct", "with_cpu_saved_pct" if "with_cpu_saved_pct" in mc_df else "net_saved_pct", "macs_saved_pct", "mj_per_sample"]].round(2)
        lines += ["## Measured savings (1-point tolerance; paired with B0 within each repeat)", "", m1.to_markdown(index=False) if _has_tabulate() else m1.to_string(index=False), ""]
        if "real_minus_offline_pp" in rv:
            worst = rv.real_minus_offline_pp.abs().max()
            lines += [f"Real versus offline accuracy, largest gap: {worst:.2f} points" + (" - INVESTIGATE: offline results may not match the real pipeline." if worst > 1.5 else "."), ""]
    if rv is not None and os.path.exists(real_path):
        raw2 = pd.read_csv(real_path)
        if "gpu_sw_thermal_slowdown" in raw2:
            gp = raw2[raw2.gpu_sw_thermal_slowdown.notna()].copy(); gp["slow"] = gp.gpu_sw_thermal_slowdown.astype(bool)
            th = gp.groupby(["hw", "batch"]).agg(slowdown_frac=("slow", "mean"), temp_c=("gpu_temp_c", "median"), clock_mhz=("gpu_clock_mhz", "median"), power_w=("gpu_power_w", "median")).round(2).reset_index()
            lines += ["## GPU thermal state during real runs", "", th.to_markdown(index=False) if _has_tabulate() else th.to_string(index=False), ""]
    cp = os.path.join(RESULTS, "float32_cooled_summary.csv")
    if os.path.exists(cp):
        lines += ["## GPU float32 with a cool-down before each run (see scripts/float32_cooled.py)", "", pd.read_csv(cp)[["batch", "family", "design", "tau", "kind", "acc_diff_pp", "measured_saved_pct", "measured_saved_min", "measured_saved_max", "time_saved_pct", "mj_per_sample"]].round(2).to_markdown(index=False), ""]
    lines += ["The float32 cost table (`cuda_laptopfp32`) was measured while throttling and is unreliable; quote its real runs only.", ""]
    lines += ["Energy compares only within one meter. Check `energy_tables.json` for the meter behind each regime.", ""]
    open(os.path.join(RESULTS, "SUMMARY.md"), "w").write("\n".join(lines))
    print("\n".join(lines))

def _has_tabulate():
    try:
        import tabulate  # noqa
        return True
    except ImportError:
        return False

def package():
    # Everything needed for the paper, none of the bulky regenerable features.
    path = os.path.join(WORK, "green_vit_results.zip")
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(os.listdir(RESULTS)):
            z.write(os.path.join(RESULTS, f), arcname=os.path.join("results", f))
    print(f"packaged {path} ({os.path.getsize(path) / 1e6:.1f} MB)")
    if ON_KAGGLE:
        print("Download from the Output tab of the saved version, or: kaggle kernels output <user>/<notebook-slug> -p ./outputs")

## 11. Run
Use **Save Version > Save & Run All** on Kaggle. Re-running continues from where it stopped.

In [ ]:
if "extract" in STAGES:
    with phase("extract", DEV):
        stage_extract()

In [ ]:
if "train" in STAGES:
    with phase("train", DEV):
        stage_train()

In [ ]:
if "points" in STAGES:
    stage_points()

In [ ]:
if "energy" in STAGES:
    stage_energy()

In [ ]:
if "select" in STAGES:
    stage_select()

In [ ]:
if "real" in STAGES:
    stage_real()

In [ ]:
if "report" in STAGES:
    stage_report()
package()